In [2]:
%pylab inline
import os
import numpy as np
import astropy.io.fits as pyfits
import astropy.table as astTable
import sim_analysis_utilities as utilities
import smatch
simDir='./wrkDir/s19/s19-1/anaCat'
dbDir='./wrkDir/database/COSMOS_HSC_S19/'

Populating the interactive namespace from numpy and matplotlib


# Match to COSMOS photo-z

In [ ]:
# Read the COSMOS catalog
cosmoname= '/work/xiangchong.li/work/galsim_train/COSMOS_25.2_training_sample/real_galaxy_catalog_25.2_fits.fits' 
assert os.path.isfile(cosmoname)
cosmocat=astTable.Table.read(cosmoname)
for fieldname in utilities.field_names:
    simFname=   os.path.join(simDir,'%s/%s.fits' %(fieldname,fieldname))
    assert os.path.isfile(simFname)
    
    simDat=astTable.Table.read(simFname)
    catOut=astTable.join(simDat,cosmocat,'IDENT','left')
    # Make sure the index is the same as the origin catalog
    catOut.sort(['isim','paired','ipos'])
    outFname=os.path.join(simDir,'%s/%s-2.fits' %(fieldname,fieldname))  
    catOut.write(outFname,overwrite=True)

False


False


False


False


# Match to HSC photo-z

In [4]:
hscCname=os.path.join(dbDir,'COSMOS.fits')
hscCat=pyfits.getdata(hscCname)

In [7]:
mask=(~np.isnan(hscCat['mizuki_photoz_best']))
hscCat=hscCat[mask]
zbest=hscCat['mizuki_photoz_best']
ra=hscCat['i_ra']
dec=hscCat['i_dec']
print(len(hscCat))

3988160


In [8]:
for fieldname in utilities.field_names:
    simFname=   os.path.join(simDir,'%s/%s-2.fits' %(fieldname,fieldname))
    assert os.path.isfile(simFname)
    simDat=astTable.Table.read(simFname)
    match   =   smatch.match(ra,dec,0.5/3600.,\
        simDat['RA'],simDat['DEC'],maxmatch=1)      # For a galaxy in sim find the closest one in HSC Wide field
    izbest=np.zeros(len(simDat))
    izbest[:]=np.NaN
    izbest[match['i2']]=zbest[match['i1']]
    simDat['mizuki_photoz_best']=izbest
    simDat.write(simFname,overwrite=True)

# Match to New COSMOS photo-z

In [1]:
cosmoName='/work/xiangchong.li/work/COSMOS/COSMOS2015_Laigle+_v1.1.fits.gz'
cosmoCat=pyfits.getdata(cosmoName)
mask=(~np.isnan(cosmoCat['PHOTOZ']))
cosmoCat=cosmoCat[mask]

In [2]:
ra=cosmoCat['ALPHA_J2000']
dec=cosmoCat['DELTA_J2000']
zbest=cosmoCat['PHOTOZ']

In [7]:
for fieldname in utilities.field_names:
    print('begin field: %s' %fieldname)
    simFname=   os.path.join(simDir,'%s/%s-2.fits' %(fieldname,fieldname))
    assert os.path.isfile(simFname)
    simDat=astTable.Table.read(simFname)
    match   =   smatch.match(ra,dec,0.5/3600.,\
        simDat['RA'],simDat['DEC'],maxmatch=1)      # For a galaxy in sim find the closest one in HSC Wide field
    izbest=np.zeros(len(simDat))
    izbest[:]=np.NaN
    izbest[match['i2']]=zbest[match['i1']]
    simDat['COSMOS15_Z']=izbest
    simDat.write(simFname,overwrite=True)
    print('finish field: %s' %fieldname)

begin field: XMM


finish field: XMM
begin field: GAMA09H
finish field: GAMA09H
begin field: WIDE12H


finish field: WIDE12H
begin field: GAMA15H


finish field: GAMA15H
begin field: VVDS


finish field: VVDS
begin field: HECTOMAP


finish field: HECTOMAP
